# Preparação de dados — atualização

Esta célula integra melhorias do script de coleta do IBGE: timeout, `raise_for_status()`, uso de `Path`, criação do diretório de destino, e gravação do CSV com `;` e `utf-8`.

In [5]:
import requests
import pandas as pd
from pathlib import Path
from IPython.display import display

# Se o notebook estiver dentro da pasta notebooks/
PASTA_RAIZ = Path.cwd().resolve().parent

PASTA_RAW = PASTA_RAIZ / "data" / "raw"
PASTA_RAW.mkdir(parents=True, exist_ok=True)


def coletar_dados_ibge_sidra():
    print("Coletando dados da API SIDRA/IBGE...")

    # Tabela 4714 - População residente
    # n6/all = todos os municípios
    # p/2022 = período 2022
    url = (
        "https://apisidra.ibge.gov.br/values/"
        "t/4714"
        "/n6/all"
        "/v/93"
        "/p/2022"
    )

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    dados = response.json()

    # A primeira linha costuma ser cabeçalho
    df = pd.DataFrame(dados[1:])

    print("Colunas retornadas pela API:")
    print(df.columns.tolist())

    # Na API SIDRA, normalmente:
    # D1C = código do município
    # D1N = nome do município
    # V = valor
    df = df.rename(
        columns={
            "D1C": "CO_MUNICIPIO_ESC",
            "D1N": "NO_MUNICIPIO_IBGE",
            "V": "POPULACAO_MUNICIPIO"
        }
    )

    df = df[
        [
            "CO_MUNICIPIO_ESC",
            "NO_MUNICIPIO_IBGE",
            "POPULACAO_MUNICIPIO"
        ]
    ]

    df["CO_MUNICIPIO_ESC"] = pd.to_numeric(
        df["CO_MUNICIPIO_ESC"],
        errors="coerce"
    ).astype("Int64")

    df["POPULACAO_MUNICIPIO"] = (
        df["POPULACAO_MUNICIPIO"]
        .astype(str)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

    df["POPULACAO_MUNICIPIO"] = pd.to_numeric(
        df["POPULACAO_MUNICIPIO"],
        errors="coerce"
    ).astype("Int64")

    return df


df_ibge = coletar_dados_ibge_sidra()

caminho_salvamento = PASTA_RAW / "ibge_populacao_municipios_2022.csv"

df_ibge.to_csv(
    caminho_salvamento,
    index=False,
    sep=";",
    encoding="utf-8"
)

print(f"Sucesso! {len(df_ibge)} municípios coletados.")
print(f"Arquivo salvo em: {caminho_salvamento}")

display(df_ibge.head())

Coletando dados da API SIDRA/IBGE...
Colunas retornadas pela API:
['NC', 'NN', 'MC', 'MN', 'V', 'D1C', 'D1N', 'D2C', 'D2N', 'D3C', 'D3N']
Sucesso! 5570 municípios coletados.
Arquivo salvo em: C:\Dash-research\data\raw\ibge_populacao_municipios_2022.csv


,CO_MUNICIPIO_ESC,NO_MUNICIPIO_IBGE,POPULACAO_MUNICIPIO
0,1100015,Alta Floresta D'Oeste - RO,21494
1,1100023,Ariquemes - RO,96833
2,1100031,Cabixi - RO,5351
3,1100049,Cacoal - RO,86887
4,1100056,Cerejeiras - RO,15890
